# 04 — Leakage-Controlled Split Protocol Design

Builds and audits VTUAD split definitions. **No model training.**

Protocols: `random` (ShipNN-style baseline), `mmsi`, `session`, `temporal`, `scenario`.

MMSI `0` = background / missing identity — not an individual vessel.

In [ ]:
from pathlib import Path
import sys
import json

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.splits import build_all_protocols, load_splits_config, prepare_catalog
from scripts.verify.audit_splits import audit_protocol, overlap_matrix, protocol_statistics

cfg = load_splits_config()
print('seed', cfg['seed'], 'ratios', cfg['ratios'])
catalog = prepare_catalog()
print('catalog', len(catalog), 'ships', catalog.loc[catalog.MMSI.fillna(0).astype(int)!=0,'MMSI'].nunique())

In [ ]:
# Rebuild all split CSVs (writes under data/processed/splits/)
results = build_all_protocols(cfg)
print('temporal boundaries', results.get('temporal_boundaries'))
for name, df in results['protocols'].items():
    print(name, df['split'].value_counts().to_dict())

In [ ]:
audits = []
for name, df in results['protocols'].items():
    a = audit_protocol(df, name)
    audits.append(a)
    print(f"{name}: MMSI_leak={a['mmsi_leakage']} session_leak={a['session_leakage']} "
          f"ships={a['train']['ships']}/{a['val']['ships']}/{a['test']['ships']}")

summary = pd.DataFrame([
    {
        'protocol': a['protocol'],
        'train_ships': a['train']['ships'],
        'val_ships': a['val']['ships'],
        'test_ships': a['test']['ships'],
        'mmsi_leakage': a['mmsi_leakage'],
        'session_leakage': a['session_leakage'],
        'mmsi_train_test_overlap': a['mmsi_overlap']['train_test'],
        'session_train_test_overlap': a['session_overlap']['train_test'],
        'sha256_train_test_overlap': a['sha256_overlap']['train_test'],
    }
    for a in audits
])
summary

In [ ]:
# Overlap matrices
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
pairs = [
    ('random', 'MMSI', True, 'Random MMSI |A∩B|'),
    ('mmsi', 'MMSI', True, 'MMSI protocol |A∩B|'),
    ('random', 'sub_init', False, 'Random session |A∩B|'),
    ('session', 'sub_init', False, 'Session protocol |A∩B|'),
]
for ax, (proto, col, nz, title) in zip(axes.ravel(), pairs):
    mat = overlap_matrix(results['protocols'][proto], col, nonzero_only=nz)
    im = ax.imshow(mat.values.astype(float), cmap='Reds')
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(mat.columns); ax.set_yticklabels(mat.index)
    ax.set_title(title)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, int(mat.values[i, j]), ha='center', va='center')
plt.tight_layout()
plt.show()

In [ ]:
# Class / ship counts per protocol
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, proto in zip(axes, ['random', 'mmsi', 'session']):
    df = results['protocols'][proto]
    ct = df.groupby(['split', 'label']).size().unstack(fill_value=0)
    ct.loc[['train','val','test']].plot(kind='bar', stacked=True, ax=ax, title=f'{proto} class mix')
plt.tight_layout()
plt.show()

## Interpretation

| Protocol | Role |
|---|---|
| **random** | ShipNN-style baseline — leaks ships/sessions |
| **mmsi** | Primary ship-independent protocol |
| **session** | Secondary — blocks neighboring-clip leakage |
| **temporal** | Chronological stress test |
| **scenario** | Cross-distance generalization |

Fingerprinting note: MMSI-disjoint classification ≠ closed-set ID of the same ships. Enrollment/query verification comes later.

See `reports/split_audit.md` for the master report.